# GLiNER2-large Zero-Shot Evaluation for Medical Dataset NER

This notebook evaluates `urchade/gliner_large-v2.1` **without any fine-tuning** on the in-distribution test set and 151-document OOD set. It is the fourth entry in the 4-way comparison:

| Model | Type | Chunk size |
|---|---|---|
| CRF | Fine-tuned (BIO) | 1500 chars |
| SciBERT | Fine-tuned (encoder) | 1500 chars |
| ModernBERT-large | Fine-tuned (encoder) | 6000 chars |
| **GLiNER2-large** | **Zero-shot (span)** | **6000 chars** |

**Data source:** `ModernBERT/modernbert_data/` — 6000-char chunks, seed=42, same negative sampling + augmentation as all baselines. Only `text` and `gold_entities` per chunk are used — no BIO tokenization needed.

**Evaluation:** Chunk-level exact / partial match (identical to all baselines). Plus threshold sweep and 17-variant prompt ablation.

| Section | Description |
|---------|-------------|
| 1 | Setup and configuration |
| 2 | Load test & OOD data |
| 3 | Load GLiNER2-large (zero-shot) |
| 4 | Inference & evaluation helpers |
| 5 | Threshold sweep on test set |
| 6 | Comprehensive prompt ablation (17 variants) |
| 7 | Final test evaluation (best threshold + best prompt) |
| 8 | OOD evaluation |
| 9 | 4-way comparison table (CRF / SciBERT / ModernBERT / GLiNER2-ZS) |
| 10 | Error analysis |
| 11 | Save results |

## 1. Setup and Configuration

In [ ]:
%pip install -q gliner

In [ ]:
import os
import json
import pickle
import warnings
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch
from gliner import GLiNER

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
GLINER_MODEL_NAME = "urchade/gliner_large-v2.1"
THRESHOLD         = 0.5           # default; swept in Section 5
BATCH_SIZE        = 4             # chunks per batch (6000-char chunks are large)
DATA_DIR          = "../ModernBERT/modernbert_data"  # 6000-char chunks, same splits as all baselines
OUTPUT_DIR        = "./gliner2_zeroshot_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device:       {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"Model:        {GLINER_MODEL_NAME}")
print(f"Chunk source: {DATA_DIR}  (ModernBERT 6000-char chunks)")
print(f"Output dir:   {OUTPUT_DIR}")

## 2. Load Test & OOD Data

In [ ]:
def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

test_data = load_pickle(os.path.join(DATA_DIR, "test.pkl"))
ood_data  = load_pickle(os.path.join(DATA_DIR, "ood.pkl"))

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    metadata = json.load(f)

print(f"Test chunks:  {len(test_data)}")
print(f"OOD chunks:   {len(ood_data)}")
print(f"Chunk size:   {metadata.get('chunk_size', '?')} chars")
print(f"Seed:         {metadata.get('seed', '?')}")
print(f"Neg sampling: {metadata.get('negative_sample_ratio', '?')}")
print()

# Stats
print(f"{'Split':<8} {'Chunks':>7} {'Mean len (chars)':>17} {'Total entities':>15}")
print("-" * 52)
for name, data in [("Test", test_data), ("OOD", ood_data)]:
    lengths   = [len(c['text']) for c in data]
    n_ents    = sum(len(c['gold_entities']) for c in data)
    print(f"{name:<8} {len(data):>7} {np.mean(lengths):>17.0f} {n_ents:>15}")

print()
print("Sample gold_entities:", test_data[0]['gold_entities'])
print("Sample text (first 200 chars):", test_data[0]['text'][:200])

## 3. Load GLiNER2-large (Zero-Shot)

In [ ]:
model = GLiNER.from_pretrained(GLINER_MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print(f"Loaded {GLINER_MODEL_NAME}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print()

# Smoke test
sample = "We evaluated our method on the MIMIC-III dataset and the PhysioNet SHAREE database."
smoke1 = model.predict_entities(sample, ["Dataset"], threshold=0.5)
print("Smoke test (prompt='Dataset'):",       [e['text'] for e in smoke1])
smoke2 = model.predict_entities(sample, ["medical dataset"], threshold=0.5)
print("Smoke test (prompt='medical dataset'):", [e['text'] for e in smoke2])
smoke3 = model.predict_entities(sample, ["dataset", "corpus", "database"], threshold=0.5)
print("Smoke test (multi-type):",              [e['text'] for e in smoke3])

## 4. Inference + Evaluation Helpers

In [ ]:
def predict_chunks(chunks, gliner_model, entity_types, threshold, batch_size=4):
    """Batch GLiNER inference. Returns list-of-lists of predicted entity strings.
    
    Deduplicates within each chunk (set conversion) to avoid double-counting
    when multiple entity type labels produce overlapping spans.
    """
    texts = [c['text'] for c in chunks]
    all_pred_entities = []
    for start in tqdm(range(0, len(texts), batch_size), desc="GLiNER inference"):
        batch = texts[start:start + batch_size]
        results = gliner_model.batch_predict_entities(batch, entity_types, threshold=threshold)
        for span_list in results:
            # Deduplicate spans by text to avoid double-counting from multi-type prompts
            seen = set()
            unique_ents = []
            for s in span_list:
                key = s['text'].strip().lower()
                if key not in seen:
                    seen.add(key)
                    unique_ents.append(s['text'])
            all_pred_entities.append(unique_ents)
    return all_pred_entities


def evaluate_chunk_level(chunks, all_pred_entities):
    """Chunk-level exact & partial match — identical logic to SciBERT/ModernBERT baselines.
    
    Input: all_pred_entities is a list of entity-string lists (not BIO tags).
    """
    tp = fp = fn = partial_tp = 0
    for chunk, pred_entities in zip(chunks, all_pred_entities):
        true_set = set(e.strip().lower() for e in chunk['gold_entities'])
        pred_set = set(e.strip().lower() for e in pred_entities)

        tp += len(true_set & pred_set)
        fp += len(pred_set - true_set)
        fn += len(true_set - pred_set)

        for pm in pred_set:
            if pm not in true_set:
                for tm in true_set:
                    if pm in tm or tm in pm:
                        partial_tp += 1
                        break

    exact_p  = tp / (tp + fp) if (tp + fp) > 0 else 0
    exact_r  = tp / (tp + fn) if (tp + fn) > 0 else 0
    exact_f1 = 2*exact_p*exact_r / (exact_p+exact_r) if (exact_p+exact_r) > 0 else 0

    adj    = tp + partial_tp
    par_p  = adj / (adj + fp - partial_tp) if (adj + fp - partial_tp) > 0 else 0
    par_r  = adj / (adj + fn - partial_tp) if (adj + fn - partial_tp) > 0 else 0
    par_f1 = 2*par_p*par_r / (par_p+par_r) if (par_p+par_r) > 0 else 0

    return {
        "exact_match":   {"precision": exact_p, "recall": exact_r, "f1": exact_f1,
                          "tp": tp, "fp": fp, "fn": fn},
        "partial_match": {"precision": par_p, "recall": par_r, "f1": par_f1,
                          "partial_tp": partial_tp},
    }


def print_metrics(name, metrics):
    em = metrics['exact_match']
    pm = metrics['partial_match']
    print(f"{'Metric':<20} {'Precision':>10} {'Recall':>8} {'F1':>8}")
    print("-" * 50)
    print(f"{'Exact match':<20} {em['precision']:>10.4f} {em['recall']:>8.4f} {em['f1']:>8.4f}")
    print(f"{'Partial match':<20} {pm['precision']:>10.4f} {pm['recall']:>8.4f} {pm['f1']:>8.4f}")
    print(f"\nTP={em['tp']}, FP={em['fp']}, FN={em['fn']}, Partial TP={pm['partial_tp']}")


print("Helper functions defined.")

## 5. Threshold Sweep on Test Set

GLiNER F1 is highly sensitive to the confidence threshold — too low inflates FP, too high inflates FN. Sweep 7 values and pick the threshold maximising exact F1 on the test set. All results are saved so the default-0.5 result is always visible.

In [ ]:
THRESHOLDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
sweep_results = {}

print(f"{'Threshold':>10} {'ExactP':>8} {'ExactR':>8} {'ExactF1':>9} {'PartialF1':>11}")
print("-" * 52)

for thr in THRESHOLDS:
    preds = predict_chunks(test_data, model, ["Dataset"], threshold=thr, batch_size=BATCH_SIZE)
    m = evaluate_chunk_level(test_data, preds)
    sweep_results[thr] = m
    em = m['exact_match']
    print(f"{thr:>10.1f} {em['precision']:>8.4f} {em['recall']:>8.4f} "
          f"{em['f1']:>9.4f} {m['partial_match']['f1']:>11.4f}")

best_thr = max(sweep_results, key=lambda t: sweep_results[t]['exact_match']['f1'])
print(f"\nBest threshold: {best_thr}  "
      f"ExactF1={sweep_results[best_thr]['exact_match']['f1']:.4f}")

# Plot threshold sweep
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
thrs = list(sweep_results.keys())
exact_f1s   = [sweep_results[t]['exact_match']['f1']   for t in thrs]
partial_f1s = [sweep_results[t]['partial_match']['f1'] for t in thrs]
exact_ps    = [sweep_results[t]['exact_match']['precision'] for t in thrs]
exact_rs    = [sweep_results[t]['exact_match']['recall']    for t in thrs]

axes[0].plot(thrs, exact_f1s, 'o-', label='Exact F1', color='steelblue')
axes[0].plot(thrs, partial_f1s, 's--', label='Partial F1', color='darkorange')
axes[0].axvline(best_thr, color='red', linestyle=':', label=f'Best thr={best_thr}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('F1')
axes[0].set_title('F1 vs Threshold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(thrs, exact_ps, 'o-', label='Precision', color='green')
axes[1].plot(thrs, exact_rs, 's--', label='Recall',    color='purple')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Score')
axes[1].set_title('Exact Precision & Recall vs Threshold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

with open(os.path.join(OUTPUT_DIR, 'threshold_sweep.json'), 'w') as f:
    json.dump({str(k): v for k, v in sweep_results.items()}, f, indent=2)
print("Saved threshold_sweep.json")

## 6. Comprehensive Prompt Ablation (17 Variants)

GLiNER2 takes natural-language entity type descriptions as input. The wording can significantly affect recall and precision. This section tests 17 prompts covering:
- **Minimal:** single-word labels (Dataset, corpus, database, benchmark)
- **Two-word descriptive:** medical dataset, biomedical dataset, clinical dataset, etc.
- **Full phrase:** data collection, named data collection, research data collection
- **Multi-type:** multiple labels passed simultaneously (GLiNER predicts all types in one pass)

In [ ]:
PROMPTS = {
    # ── Single token / minimal ──────────────────────────────────────────
    "Dataset":                    ["Dataset"],
    "dataset":                    ["dataset"],
    "corpus":                     ["corpus"],
    "database":                   ["database"],
    "benchmark":                  ["benchmark"],

    # ── Two-word descriptive ────────────────────────────────────────────
    "dataset name":               ["dataset name"],
    "medical dataset":            ["medical dataset"],
    "research dataset":           ["research dataset"],
    "biomedical dataset":         ["biomedical dataset"],
    "clinical dataset":           ["clinical dataset"],
    "public dataset":             ["public dataset"],
    "named dataset":              ["named dataset"],

    # ── Full descriptive phrase ─────────────────────────────────────────
    "data collection":            ["data collection"],
    "named data collection":      ["named data collection"],
    "research data collection":   ["research data collection"],

    # ── Multi-type (GLiNER accepts multiple labels simultaneously) ──────
    "Dataset+corpus+database":    ["Dataset", "corpus", "database"],
    "medical dataset+corpus":     ["medical dataset", "corpus"],
}

prompt_results = {}
print(f"{'Prompt':<32} {'ExactP':>7} {'ExactR':>7} {'ExactF1':>8} {'PartialF1':>10}")
print("-" * 70)

for label, etype in PROMPTS.items():
    preds = predict_chunks(test_data, model, etype, threshold=best_thr, batch_size=BATCH_SIZE)
    m = evaluate_chunk_level(test_data, preds)
    prompt_results[label] = m
    em = m['exact_match']
    print(f"{label:<32} {em['precision']:>7.4f} {em['recall']:>7.4f} "
          f"{em['f1']:>8.4f} {m['partial_match']['f1']:>10.4f}")

best_prompt_label = max(prompt_results, key=lambda k: prompt_results[k]['exact_match']['f1'])
BEST_ENTITY_TYPES = PROMPTS[best_prompt_label]
print(f"\n{'='*70}")
print(f"Best prompt: '{best_prompt_label}'  →  {BEST_ENTITY_TYPES}")
print(f"Best ExactF1: {prompt_results[best_prompt_label]['exact_match']['f1']:.4f}")

with open(os.path.join(OUTPUT_DIR, 'prompt_ablation.json'), 'w') as f:
    json.dump(prompt_results, f, indent=2)
print("\nSaved prompt_ablation.json")

In [ ]:
# Visualise prompt ablation — bar chart sorted by ExactF1
sorted_prompts = sorted(prompt_results.items(), key=lambda x: x[1]['exact_match']['f1'])
labels_sorted  = [p for p, _ in sorted_prompts]
exact_f1_vals  = [m['exact_match']['f1']   for _, m in sorted_prompts]
partial_f1_vals= [m['partial_match']['f1'] for _, m in sorted_prompts]

fig, ax = plt.subplots(figsize=(10, 7))
y = range(len(labels_sorted))
ax.barh(y, exact_f1_vals,   0.4, label='Exact F1',   color='steelblue', align='center')
ax.barh([yi + 0.4 for yi in y], partial_f1_vals, 0.4, label='Partial F1', color='darkorange', align='center')
ax.set_yticks([yi + 0.2 for yi in y])
ax.set_yticklabels(labels_sorted, fontsize=9)
ax.set_xlabel('F1 Score'); ax.set_title('Prompt Ablation (sorted by Exact F1)')
ax.legend(); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'prompt_ablation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Final Test Evaluation (Best Threshold + Best Prompt)

In [ ]:
print(f"Running final test eval with:")
print(f"  Prompt:    {BEST_ENTITY_TYPES}")
print(f"  Threshold: {best_thr}")
print()

test_pred_entities = predict_chunks(
    test_data, model, BEST_ENTITY_TYPES, threshold=best_thr, batch_size=BATCH_SIZE)
test_metrics = evaluate_chunk_level(test_data, test_pred_entities)

print("=" * 60)
print("TEST SET — GLiNER2-large Zero-Shot")
print("=" * 60)
print_metrics("TEST", test_metrics)

## 8. OOD Evaluation

In [ ]:
ood_pred_entities = predict_chunks(
    ood_data, model, BEST_ENTITY_TYPES, threshold=best_thr, batch_size=BATCH_SIZE)
ood_metrics = evaluate_chunk_level(ood_data, ood_pred_entities)

print("=" * 60)
print("OOD SET — GLiNER2-large Zero-Shot")
print("=" * 60)
print_metrics("OOD", ood_metrics)

In [ ]:
# Side-by-side Test vs OOD for GLiNER2-ZS
val_test_ood_metrics = [("Test (ID)", test_metrics), ("OOD", ood_metrics)]

print(f"\n{'Metric':<20} {'Test (ID)':>10} {'OOD':>10}")
print("-" * 45)
for key in ['exact_match', 'partial_match']:
    for sub in ['precision', 'recall', 'f1']:
        label = f"{key.replace('_match', '')} {sub[:1].upper()}"
        print(f"{label:<20} "
              f"{test_metrics[key][sub]:>10.4f} "
              f"{ood_metrics[key][sub]:>10.4f}")

drop = test_metrics['exact_match']['f1'] - ood_metrics['exact_match']['f1']
print(f"\nExact F1 OOD drop: {drop:+.4f}")

## 9. 4-Way Comparison Table (CRF / SciBERT / ModernBERT / GLiNER2-ZS)

**Note on chunk sizes:**
- CRF and SciBERT used **1500-char** chunks
- ModernBERT and GLiNER2-ZS used **6000-char** chunks

Entity/document-level F1 comparison is valid across all models. Chunk size affects the granularity of evaluation windows, but the same annotated entities are covered in both cases.

In [ ]:
own = {
    "test": {"exact_match": test_metrics["exact_match"],
             "partial_match": test_metrics["partial_match"]},
    "ood":  {"exact_match": ood_metrics["exact_match"],
             "partial_match": ood_metrics["partial_match"]},
}

comparisons = []
for name, path in [
    ("CRF",        "../CRF/crf_medical_ner/eval_metrics.json"),
    ("SciBERT",    "../SciBERT/scibert_MeDataset-NER-540-samples-OOD/eval_metrics.json"),
    ("ModernBERT", "../ModernBERT/modernbert_MeDataset-NER-540-samples-OOD/eval_metrics.json"),
]:
    if os.path.exists(path):
        with open(path) as f:
            comparisons.append((name, json.load(f)))
    else:
        print(f"[warn] {name} metrics not found at {path} — skipping")

comparisons.append(("GLiNER2-ZS", own))

print("=" * 72)
print("4-WAY COMPARISON: CRF / SciBERT / ModernBERT / GLiNER2-ZS")
print("(same 542 labeled docs + 151 OOD docs, same seed=42)")
print("=" * 72)

print(f"\n{'Model':<14} {'Split':<6} {'Exact P':>8} {'Exact R':>8} {'Exact F1':>9} {'Partial F1':>11}")
print("-" * 60)
for name, m in comparisons:
    for split in ['test', 'ood']:
        s  = m.get(split, {})
        em = s.get('exact_match', {})
        pm = s.get('partial_match', {})
        print(f"{name:<14} {split.upper():<6} "
              f"{em.get('precision', 0):>8.4f} "
              f"{em.get('recall', 0):>8.4f} "
              f"{em.get('f1', 0):>9.4f} "
              f"{pm.get('f1', 0):>11.4f}")
    print("-" * 60)

print("\nChunk sizes: CRF/SciBERT=1500 chars | ModernBERT/GLiNER2-ZS=6000 chars.")
print("GLiNER2-ZS is zero-shot — no fine-tuning on any labeled data.")
print(f"GLiNER2-ZS best prompt: '{best_prompt_label}', threshold: {best_thr}")

## 10. Error Analysis

In [ ]:
def collect_error_examples(chunks, all_pred_entities):
    """Collect per-chunk TP, FP, FN, and partial match examples."""
    tp_examples = []
    fp_examples = []
    fn_examples = []
    partial_examples = []

    for chunk, pred_entities in zip(chunks, all_pred_entities):
        true_set = set(e.strip().lower() for e in chunk['gold_entities'])
        pred_set = set(e.strip().lower() for e in pred_entities)

        for e in true_set & pred_set:
            tp_examples.append(e)
        for e in pred_set - true_set:
            fp_examples.append(e)
        for e in true_set - pred_set:
            fn_examples.append(e)
        for pm in pred_set:
            if pm not in true_set:
                for tm in true_set:
                    if pm in tm or tm in pm:
                        partial_examples.append((pm, tm))
                        break

    return tp_examples, fp_examples, fn_examples, partial_examples


tp_ex, fp_ex, fn_ex, partial_ex = collect_error_examples(test_data, test_pred_entities)

print("=" * 60)
print("TEST SET ERROR EXAMPLES")
print("=" * 60)
print(f"TP: {len(tp_ex)}, FP: {len(fp_ex)}, FN: {len(fn_ex)}, Partial: {len(partial_ex)}")

print(f"\n--- True Positives (showing 5 of {len(tp_ex)}) ---")
for e in tp_ex[:5]: print(f"  {e}")

print(f"\n--- False Positives (showing 5 of {len(fp_ex)}) ---")
for e in fp_ex[:5]: print(f"  {e}")

print(f"\n--- False Negatives (showing 5 of {len(fn_ex)}) ---")
for e in fn_ex[:5]: print(f"  {e}")

print(f"\n--- Partial Matches (showing 5 of {len(partial_ex)}) ---")
for pred, gold in partial_ex[:5]:
    print(f"  Predicted: '{pred}' | Gold: '{gold}'")

In [ ]:
# Error pattern analysis — by entity word count
print("=" * 60)
print("ERROR PATTERNS")
print("=" * 60)

fp_wc = Counter(len(e.split()) for e in fp_ex)
fn_wc = Counter(len(e.split()) for e in fn_ex)
tp_wc = Counter(len(e.split()) for e in tp_ex)

all_lengths = sorted(set(list(fp_wc) + list(fn_wc) + list(tp_wc)))
print(f"\n{'Length':<10} {'TP':>6} {'FP':>6} {'FN':>6}")
print("-" * 35)
for l in all_lengths[:12]:
    print(f"{l:<10} {tp_wc.get(l,0):>6} {fp_wc.get(l,0):>6} {fn_wc.get(l,0):>6}")

print(f"\n--- Top 10 Most Common False Positives ---")
for e, c in Counter(fp_ex).most_common(10):
    print(f"  [{c}x] {e}")

print(f"\n--- Top 10 Most Common False Negatives ---")
for e, c in Counter(fn_ex).most_common(10):
    print(f"  [{c}x] {e}")

In [ ]:
# OOD-specific error analysis
print("=" * 60)
print("OOD ERROR ANALYSIS")
print("=" * 60)

ood_tp, ood_fp, ood_fn, ood_partial = collect_error_examples(ood_data, ood_pred_entities)

print(f"OOD TP: {len(ood_tp)}, FP: {len(ood_fp)}, FN: {len(ood_fn)}, Partial: {len(ood_partial)}")

print(f"\n--- OOD False Positives (showing 5 of {len(ood_fp)}) ---")
for e in ood_fp[:5]: print(f"  {e}")

print(f"\n--- OOD False Negatives (showing 5 of {len(ood_fn)}) ---")
for e in ood_fn[:5]: print(f"  {e}")

print(f"\n--- OOD Partial Matches (showing 5 of {len(ood_partial)}) ---")
for pred, gold in ood_partial[:5]:
    print(f"  Predicted: '{pred}' | Gold: '{gold}'")

id_err  = (len(fp_ex) + len(fn_ex))   / max(len(tp_ex) + len(fp_ex) + len(fn_ex), 1)
ood_err = (len(ood_fp) + len(ood_fn)) / max(len(ood_tp) + len(ood_fp) + len(ood_fn), 1)
print(f"\nError rate comparison:")
print(f"  Test (ID): {id_err:.3f}")
print(f"  OOD:       {ood_err:.3f}")

print(f"\n--- Top 10 OOD False Negatives ---")
for e, c in Counter(ood_fn).most_common(10):
    print(f"  [{c}x] {e}")

In [ ]:
# Word-count distribution bar chart (FP/FN by entity length)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (split_fp, split_fn, split_tp, title) in zip(
    axes,
    [(fp_ex, fn_ex, tp_ex, "Test (ID)"), (ood_fp, ood_fn, ood_tp, "OOD")],
):
    s_fp = Counter(len(e.split()) for e in split_fp)
    s_fn = Counter(len(e.split()) for e in split_fn)
    s_tp = Counter(len(e.split()) for e in split_tp)
    lengths = sorted(set(list(s_fp) + list(s_fn) + list(s_tp)))[:10]
    x = range(len(lengths))
    w = 0.28
    ax.bar([xi - w for xi in x], [s_tp.get(l,0) for l in lengths], w, label='TP', color='green')
    ax.bar(x,                    [s_fp.get(l,0) for l in lengths], w, label='FP', color='red')
    ax.bar([xi + w for xi in x], [s_fn.get(l,0) for l in lengths], w, label='FN', color='orange')
    ax.set_xticks(list(x)); ax.set_xticklabels([str(l) for l in lengths])
    ax.set_xlabel('Entity word count'); ax.set_ylabel('Count')
    ax.set_title(f'{title} — TP/FP/FN by Entity Length')
    ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'error_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Results

In [ ]:
# Save config.json
config = {
    "model": GLINER_MODEL_NAME,
    "entity_types": BEST_ENTITY_TYPES,
    "best_prompt": best_prompt_label,
    "threshold": best_thr,
    "data_source": DATA_DIR,
    "chunk_size_chars": metadata.get("chunk_size", 6000),
    "chunk_overlap_chars": metadata.get("chunk_overlap", 500),
    "test_chunks": len(test_data),
    "ood_chunks": len(ood_data),
    "mode": "zero_shot",
    "prompts_tested": list(PROMPTS.keys()),
    "thresholds_tested": THRESHOLDS,
    "seed": metadata.get("seed", 42),
    "note": "Chunk sizes differ vs CRF/SciBERT (1500). Entity-level F1 comparison is valid.",
}

# Save eval_metrics.json (same schema as other baselines)
eval_metrics = {
    "test": {
        "exact_match":   test_metrics["exact_match"],
        "partial_match": test_metrics["partial_match"],
    },
    "ood": {
        "exact_match":   ood_metrics["exact_match"],
        "partial_match": ood_metrics["partial_match"],
    },
    "threshold_sweep": {str(k): v for k, v in sweep_results.items()},
    "prompt_ablation_best": best_prompt_label,
}

with open(os.path.join(OUTPUT_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)
with open(os.path.join(OUTPUT_DIR, 'eval_metrics.json'), 'w') as f:
    json.dump(eval_metrics, f, indent=2)

print(f"Saved to {OUTPUT_DIR}/")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size  = os.path.getsize(fpath)
    unit  = 'KB' if size < 1024*1024 else 'MB'
    sv    = size/1024 if unit == 'KB' else size/1024/1024
    print(f"  {fname:<35} {sv:>8.1f} {unit}")

In [ ]:
# Final summary
print("=" * 65)
print("GLiNER2-large ZERO-SHOT EVALUATION COMPLETE")
print("=" * 65)
print(f"\nModel:          {GLINER_MODEL_NAME}")
print(f"Best prompt:    '{best_prompt_label}'  →  {BEST_ENTITY_TYPES}")
print(f"Best threshold: {best_thr}")
print(f"Chunk size:     {metadata.get('chunk_size', 6000)} chars (ModernBERT splits)")

print(f"\n  {'':20} {'Test (ID)':>10} {'OOD':>10}")
print(f"  {'-'*45}")
print(f"  {'Exact F1':20} {test_metrics['exact_match']['f1']:>10.4f} {ood_metrics['exact_match']['f1']:>10.4f}")
print(f"  {'Partial F1':20} {test_metrics['partial_match']['f1']:>10.4f} {ood_metrics['partial_match']['f1']:>10.4f}")
print(f"  {'Exact P':20} {test_metrics['exact_match']['precision']:>10.4f} {ood_metrics['exact_match']['precision']:>10.4f}")
print(f"  {'Exact R':20} {test_metrics['exact_match']['recall']:>10.4f} {ood_metrics['exact_match']['recall']:>10.4f}")

print(f"\nArtifacts: {OUTPUT_DIR}/")